In [ ]:
!pip install pandas

In [2]:
import time

In [3]:
!pip install -q bs4 selenium

In [4]:
import pandas as pd
import numpy as np

from selenium import webdriver
from bs4 import BeautifulSoup

In [5]:
# Untuk jalankan bot
driver = webdriver.Chrome()

# Variabel yang dipakai untuk DataFrame
tipe_produk = []
nama_produk  = []
harga = []
harga_diskon = []
rating_produk = []

# hal digunakan untuk pindah page disini 1,9 = dari page 1 sampai page 9
for hal in range(1,9):

# Url yang dipakai untuk scraping
    url = f'https://erigostore.co.id/collections/all-product?page={hal}'

# Menjalankan Bot
    driver.get(url)

# Jeda waktu yang dipakai untuk Scraping
    time.sleep(5)

    html = driver.page_source # Mengambil data mentah dari HTML
    soup = BeautifulSoup(html, "html.parser") # Mengubah HTML mentah menjadi objek

    #Box awal
    boxes = soup.find_all('div', {'class': 'product-card-wrapper'})

    #Box Harga
    box_harga = soup.find_all('div', {'class': 'card-information'})

    for box in boxes:

        # Brand
        nama_brand = box.find('div',{'class': 'additional__product-type'})
        if nama_brand is None:
            continue
        try:
            tipe_produk.append(nama_brand.get_text().strip())
        except:
            tipe_produk.append(None)

        # NAMA BAJU
        baju = box.find('h3', {'class': 'card__heading'})
        try:
            nama_produk .append(baju.get_text().strip())
        except:
            nama_produk .append(None)

    for harga_detail in box_harga:

        # HARGA REGULER
        harga_reguler = harga_detail.find('span',{'class': 'price-item--regular'})
        # ' '.join menggabungkan kembali potongan-potongan kata menjadi satu kalimat utuh dengan spasi sebagai perekatnya. Pasangan .split() 
        try:                                                               
            pembersih_reguler = ' '.join(harga_reguler.get_text().split()) # .split = membersihkan \n, \ atau pun spasi
            harga.append(pembersih_reguler)
        except:
            harga.append('tidak ada')                       

        # HARGA DISKON
        harga_potongan = harga_detail.find('span',{'class':'price-item--last'})
        try:
            pembersih_potongan = ' '.join(harga_potongan.get_text().split())
            harga_diskon.append(pembersih_potongan)
        except:
            harga_diskon.append('Tidak Ada Diskon')       

        # RATING
        rate = harga_detail.find('p', {'class': 'rating-text'})
        try:
            rating_bersih = ' '.join(rate.get_text().split())
            rating_produk.append(rating_bersih)
        except:
            rating_produk.append('Belum Ada Rating')

# Bot mati
driver.quit()

# Memasukkan scrape tadi ke dalam DataFrame
df = pd.DataFrame({
    'tipe_produk' : tipe_produk[:50],
    'nama_produk' : nama_produk [:50],
    'harga' : harga[:50],
    'rating_produk' : rating_produk[:50]
})

# Menyimpan data hasil Scrape tadi ke bantuk .CSV
# index=False > untuk tidak menyimpan urutan angka saat dijadikan .CSV
df.to_csv('data_raw_jualan_erigo.csv', index=False)

In [ ]:
# Untuk membaca data dari hasil scrape yang sudah diubah kebentuk CSV
# Lihat lokasi penyimpanan File, dan sesuaikan
df = pd.read_csv('data_raw_jualan_erigo.csv')

In [7]:
# Cek data pada DataFrame
df

,tipe_produk,nama_produk,harga,rating_produk
0,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,Rp 103.000,Belum Ada Rating
1,Chino Pants,Erigo Chino Pants Sirius Black Unisex,Rp 183.000,Belum Ada Rating
2,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,Rp 118.000,Belum Ada Rating
3,Chino Pants,Erigo Chino Pants Light Grey Unisex,Rp 183.000,Belum Ada Rating
4,Chino Pants,Erigo Chino Pants Dark Grey Unisex,Rp 183.000,Belum Ada Rating
5,Short Shirt Pocket,Erigo Short Shirt Pocket Danvin Teracotta - Ke...,Rp 145.000,Belum Ada Rating
6,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,Rp 103.000,3.83 / 5.0
7,Chino Pants,Erigo Chino Pants Sirius Black Unisex,Rp 183.000,Belum Ada Rating
8,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,Rp 118.000,5.0 / 5.0
9,Chino Pants,Erigo Chino Pants Light Grey Unisex,Rp 183.000,1.0 / 5.0


In [8]:
# Menampilkan info Data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   tipe_produk    49 non-null     str  
 1   nama_produk    50 non-null     str  
 2   harga          50 non-null     str  
 3   rating_produk  50 non-null     str  
dtypes: str(4)
memory usage: 1.7 KB


In [9]:
# Menghapus karakter RP, (.) titik, dan mengganti ke tipe INT
df['harga'] = df['harga'].str.replace('Rp','') \
                                        .str.replace('.','') \
                                            .astype(int)

In [10]:
# Membersihkan kolom rating dengan menghapus teks "/ 5.0"
df['rating_produk'] = df['rating_produk'].str.replace(' / 5.0', '', regex=False)

# Mengganti nilai rating yang belum tersedia menjadi NaN agar mudah diproses di SQL
df['rating_produk'] = df['rating_produk'].replace('Belum Ada Rating', np.nan)

# Ubah jadi angka murni
df['rating_produk'] = df['rating_produk'].astype(float)

In [11]:
# melihat Baris dari kolom 'nama_produk' yang memiliki duplikat
df[df.duplicated(subset='nama_produk', keep='first')]

,tipe_produk,nama_produk,harga,rating_produk
6,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,103000,3.83
7,Chino Pants,Erigo Chino Pants Sirius Black Unisex,183000,NaN
8,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,118000,5.00
9,Chino Pants,Erigo Chino Pants Light Grey Unisex,183000,1.00
10,Chino Pants,Erigo Chino Pants Dark Grey Unisex,183000,NaN
11,Short Shirt Pocket,Erigo Short Shirt Pocket Danvin Teracotta - Ke...,145000,NaN
42,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,103000,3.83
43,Chino Pants,Erigo Chino Pants Sirius Black Unisex,183000,NaN
44,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,118000,5.00
45,Chino Pants,Erigo Chino Pants Light Grey Unisex,183000,1.00


In [12]:
# Menghapus data duplikat menggunakan .drop_duplicates menggunakan kolom 'nama_produk'
# keep='first' > data yang muncul pertama itu yang dijadikan data asli, selain itu hapus 
# .reset_index(drop=True) merapihkan angka yang muncul di index lama agar tau berapa total data bersih
# Menyimpan data bersih dalam variabel 'df_cleaner'
df_cleaner = df.drop_duplicates(subset='nama_produk', keep='first').reset_index(drop=True)

In [13]:
# cek kembali isi data
df_cleaner

,tipe_produk,nama_produk,harga,rating_produk
0,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,103000,NaN
1,Chino Pants,Erigo Chino Pants Sirius Black Unisex,183000,NaN
2,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,118000,NaN
3,Chino Pants,Erigo Chino Pants Light Grey Unisex,183000,NaN
4,Chino Pants,Erigo Chino Pants Dark Grey Unisex,183000,NaN
5,Short Shirt Pocket,Erigo Short Shirt Pocket Danvin Teracotta - Ke...,145000,NaN
6,Chino Pants,Erigo Chino Pants Alexa Navy Unisex,183000,4.33
7,Short Shirt,Erigo Short Shirt Othieno Army Unisex,118000,5.00
8,T-Shirt Oversize Basic,Erigo T-Shirt Basic Meghan Black Unisex,103000,2.00
9,Chino Pants,Erigo Chino Pants Caprio Brown Unisex,183000,5.00


In [14]:
# Menyimpan data yang sudah dibersihkan tadi untuk diolah ke PostgreSQL
# index=False > untuk tidak menyimpan urutan angka saat dijadikan CSV
df_cleaner.to_csv('data_clean_jualan_erigo.csv', index=False)

In [ ]:
# Import DataFrame Bersih
# Lihat lokasi penyimpanan File, dan sesuaikan
df_clean = pd.read_csv('data_clean_jualan_erigo.csv')

In [16]:
# Cek DataFrame Bersih
df_clean

,tipe_produk,nama_produk,harga,rating_produk
0,T-Shirt Oversize HD,Erigo T-Shirt Oversize Antelope Black Unisex,103000,NaN
1,Chino Pants,Erigo Chino Pants Sirius Black Unisex,183000,NaN
2,Short Shirt,Erigo Short Shirt Rayon Jazlyn Black Unisex,118000,NaN
3,Chino Pants,Erigo Chino Pants Light Grey Unisex,183000,NaN
4,Chino Pants,Erigo Chino Pants Dark Grey Unisex,183000,NaN
5,Short Shirt Pocket,Erigo Short Shirt Pocket Danvin Teracotta - Ke...,145000,NaN
6,Chino Pants,Erigo Chino Pants Alexa Navy Unisex,183000,4.33
7,Short Shirt,Erigo Short Shirt Othieno Army Unisex,118000,5.00
8,T-Shirt Oversize Basic,Erigo T-Shirt Basic Meghan Black Unisex,103000,2.00
9,Chino Pants,Erigo Chino Pants Caprio Brown Unisex,183000,5.00
